In [ ]:
# ============================================================
# STEP 1 — Caricamento, pulizia e preparazione EU-LS 2022
# Does Social Media Use Reduce Loneliness?
# ============================================================

import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.2f}'.format)

os.makedirs('output/dataset', exist_ok=True)
os.makedirs('output/figures', exist_ok=True)

# ------------------------------------------------------------------
# 1.1  CARICAMENTO
# ------------------------------------------------------------------
df_raw = pd.read_csv('eu_loneliness_survey_eu27_values.csv', low_memory=False)
print(f"Dataset grezzo: {df_raw.shape[0]} righe × {df_raw.shape[1]} colonne")

# ------------------------------------------------------------------
# 1.2  SELEZIONE COLONNE DI INTERESSE
# ------------------------------------------------------------------
COLS = ['loneliness_ucla_a', 'loneliness_ucla_b', 'loneliness_ucla_c',
        'sm_time_a', 'country', 'age', 'gender', 'education', 'income_decile']

df = df_raw[COLS].copy()

# ------------------------------------------------------------------
# 1.3  COSTRUZIONE score_UCLA (variabile dipendente)
# ------------------------------------------------------------------
# Scala UCLA 3 item, valori 1–3 (999 = non risposta)
# Score = somma dei 3 item → range 3–9 (9 = massima solitudine)

for col in ['loneliness_ucla_a', 'loneliness_ucla_b', 'loneliness_ucla_c']:
    df[col] = df[col].replace(999, np.nan)

df['score_UCLA'] = df[['loneliness_ucla_a',
                        'loneliness_ucla_b',
                        'loneliness_ucla_c']].sum(axis=1, min_count=3)

print("\n--- score_UCLA ---")
print(df['score_UCLA'].describe().round(2))
print(f"Missing: {df['score_UCLA'].isna().sum()} ({df['score_UCLA'].isna().mean()*100:.1f}%)")

# ------------------------------------------------------------------
# 1.4  COSTRUZIONE intensita_sm (variabile indipendente — ORDINALE)
# ------------------------------------------------------------------
# sm_time_a: 1–8 = livelli crescenti di tempo sui social media
#            999  = non usa social media → ricodificato come 0
# Variabile finale: 0 (non utente) + 1–8 (utente per intensità)
# Range finale: 0–8 (0 = mai, 8 = uso massimo)

df['intensita_sm'] = df['sm_time_a'].replace(999, 0)

print("\n--- intensita_sm ---")
print(df['intensita_sm'].value_counts().sort_index())
print(f"\nMedia: {df['intensita_sm'].mean():.2f}")
print(f"Missing: {df['intensita_sm'].isna().sum()}")

# ------------------------------------------------------------------
# 1.5  COSTRUZIONE fascia_eta (variabile moderatrice)
# ------------------------------------------------------------------
bins   = [15, 34, 54, 74, 120]
labels = ['16–34', '35–54', '55–74', '75+']

df['fascia_eta'] = pd.cut(df['age'], bins=bins, labels=labels, right=True)

print("\n--- Distribuzione fasce d'età ---")
print(df['fascia_eta'].value_counts().sort_index())

# ------------------------------------------------------------------
# 1.6  PULIZIA VARIABILI DI CONTROLLO
# ------------------------------------------------------------------
for col in ['gender', 'education', 'income_decile']:
    df[col] = df[col].replace(999, np.nan)

print("\n--- Missing variabili di controllo ---")
for col in ['gender', 'education', 'income_decile']:
    n = df[col].isna().sum()
    print(f"  {col:20s}: {n} ({n/len(df)*100:.1f}%)")

# ------------------------------------------------------------------
# 1.7  LISTWISE DELETION
# ------------------------------------------------------------------
COLONNE_MODELLO = ['score_UCLA', 'intensita_sm', 'age', 'fascia_eta',
                   'gender', 'education', 'income_decile', 'country']

n_prima = len(df)
df_clean = df.dropna(subset=COLONNE_MODELLO).copy()
n_dopo   = len(df_clean)

print(f"\n--- Listwise deletion ---")
print(f"Righe prima : {n_prima}")
print(f"Righe dopo  : {n_dopo}")
print(f"Rimosse     : {n_prima - n_dopo} ({(n_prima - n_dopo)/n_prima*100:.1f}%)")

# ------------------------------------------------------------------
# 1.8  DATAFRAME FINALE
# ------------------------------------------------------------------
COLONNE_FINALI = ['country', 'intensita_sm', 'score_UCLA',
                  'fascia_eta', 'age', 'gender', 'education', 'income_decile']

df_final = df_clean[COLONNE_FINALI].reset_index(drop=True)
df_final.columns = ['paese', 'intensita_sm', 'score_UCLA',
                    'fascia_eta', 'eta', 'sesso', 'education', 'income']

print("\n=== DataFrame finale ===")
print(f"Shape: {df_final.shape}")
print(f"Paesi presenti: {df_final['paese'].nunique()}")
print(f"\nAnteprima:\n{df_final.head(5)}")

# ------------------------------------------------------------------
# 1.9  GRAFICI DIAGNOSTICI
# ------------------------------------------------------------------
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Distribuzione score UCLA
axes[0].hist(df_final['score_UCLA'], bins=7, color='steelblue', edgecolor='white')
axes[0].set_title('Distribuzione score UCLA')
axes[0].set_xlabel('Score (3–9)')
axes[0].set_ylabel('Frequenza')

# Distribuzione intensita_sm (ordinale 0–8)
conteggi = df_final['intensita_sm'].value_counts().sort_index()
axes[1].bar(conteggi.index, conteggi.values, color='steelblue', edgecolor='white')
axes[1].set_title('Distribuzione intensità uso social media')
axes[1].set_xlabel('Livello (0 = non utente, 8 = uso massimo)')
axes[1].set_ylabel('Frequenza')
axes[1].set_xticks(range(9))

# Score UCLA medio per livello di intensita_sm
medie_globali = df_final.groupby('intensita_sm')['score_UCLA'].mean()
axes[2].plot(medie_globali.index, medie_globali.values,
             marker='o', color='steelblue', linewidth=2)
axes[2].set_title('Score UCLA medio per intensità social media')
axes[2].set_xlabel('Livello intensita_sm')
axes[2].set_ylabel('Score UCLA medio')
axes[2].set_xticks(range(9))

plt.tight_layout()
fig.savefig('output/figures/step1_diagnostici.png', dpi=150, bbox_inches='tight')
plt.show()

# ------------------------------------------------------------------
# 1.10  SALVATAGGIO
# ------------------------------------------------------------------
df_final.to_csv('output/dataset/eu_ls_clean.csv', index=False)
print("\nSalvato: output/dataset/eu_ls_clean.csv")
print("Salvato: output/figures/step1_diagnostici.png")